In [ ]:
# 查看当前挂载的数据集目录, 该目录下的变更重启环境后会自动还原
# View dataset directory. 
# This directory will be recovered automatically after resetting environment. 
!ls /home/aistudio/data

In [ ]:
# 查看工作区文件，该目录下除data目录外的变更将会持久保存。请及时清理不必要的文件，避免加载过慢。
# View personal work directory. 
# All changes, except /data, under this directory will be kept even after reset. 
# Please clean unnecessary files in time to speed up environment loading. 
!ls /home/aistudio

In [ ]:
# 如果需要进行持久化安装, 需要使用持久化路径, 如下方代码示例:
# If a persistence installation is required, 
# you need to use the persistence path as the following: 
!mkdir /home/aistudio/external-libraries
!pip install beautifulsoup4 -t /home/aistudio/external-libraries

In [2]:
# 同时添加如下代码, 这样每次环境(kernel)启动的时候只要运行下方代码即可: 
# Also add the following code, 
# so that every time the environment (kernel) starts, 
# just run the following code: 
import sys 
sys.path.append('/home/aistudio/external-libraries')

In [2]:
#!/bin/bash

# 解决依赖冲突的安装脚本

# 1. 首先升级pip
# ! pip install --upgrade pip

! python -m pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

# 2. 安装核心音频处理包 (使用兼容版本)
! pip install librosa soundfile resampy

# 3. 安装视频处理包 (使用兼容版本)
! pip install opencv-python  # 兼容paddlex
! pip install ffmpeg-python

# 4. 安装数据处理包 (使用兼容版本)
! pip install numpy  # 兼容paddlex
! pip install pandas 
! pip install tqdm  # 满足datasets要求

# 5. 安装文件下载包
! pip install wget gdown

# 6. 安装其他实用包 (使用兼容版本)
! pip install matplotlib
! pip install seaborn
! pip install scikit-learn

# 7. 解决特定依赖冲突
# ! pip install "aiofiles<24.0,>=22.0"  # 兼容gradio
! pip install protobuf  # 兼容note-seq

# 8. 安装配置管理包
! pip install pyyaml

# 9. 安装PaddlePaddle依赖
! pip install decorator


Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/, https://mirrors.aliyun.com/pypi/simple/
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 13.1 MB/s eta 0:00:00a 0:00:01
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-scri

In [3]:
! python -c "import paddle; print(paddle.device.is_compiled_with_cuda()); print(paddle.device.cuda.device_count())"

/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
True
1


In [4]:
!pip install transformers
!pip install datasets

Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 66.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 68.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 76.9 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 3.40.0 requires aiofiles<24.0,>=22.0, but you have aiofiles 24.1.0 which is incompatible.
paddlenlp 3.0.0b3 requires tokenizers<0.21, but you have tokenizers 0.21.1 which is incompatible.
paddlex 3.0.0b2 requires tokenizers==0.19.1, but you have to

In [4]:
from datasets import load_dataset
#! huggingface-cli login

In [5]:
import os
import json
import random
import librosa
import numpy as np
import pandas as pd
import paddle
from datasets import load_dataset
from tqdm import tqdm

# ============================================================
# 配置参数
# ============================================================
class Config:
    # 训练参数
    num_epochs = 20          # 训练轮数
    batch_size = 32          # 批次大小
    learning_rate = 0.001    # 初始学习率
    weight_decay = 1e-5      # 权重衰减
    
    # 音频处理参数
    sample_rate = 16000      # 音频采样率
    audio_length = 16000     # 音频长度（采样点数）
    num_workers = 2          # 数据加载线程数
    
    # 频谱图参数
    n_fft = 1024             # FFT窗口大小
    hop_length = 256         # 帧移
    n_mels = 64              # 梅尔滤波器数量
    
    # HTSAT 模型参数
    htsat_dim = 128          # 特征维度
    htsat_num_head = 4       # 注意力头数
    htsat_window_size = 8    # 窗口大小
    htsat_depth = 4          # Swin层数
    htsat_mlp_ratio = 4      # MLP扩展比例
    htsat_qkv_bias = True    # 是否使用QKV偏置
    htsat_drop_rate = 0.1    # Dropout率
    htsat_attn_drop_rate = 0.1 # 注意力Dropout率
    
    # 分类参数
    num_classes = 2          # 类别数

config = Config()

# ============================================================
# 加载原始数据集
# ============================================================
print("-------------加载原始数据集-------------")
raw_ds = load_dataset('parquet', data_files='/home/aistudio/train-00000-of-00001.parquet')
full_dataset = raw_ds['train']  # 提取唯一的数据集拆分

# 创建标签映射
label_map = {0: "non_violence", 1: "violence"}
dataset_path = "/home/aistudio"

# 保存标签映射
with open(os.path.join(dataset_path, "label_map.json"), 'w') as f:
    json.dump(label_map, f)

# ============================================================
# 预处理并保存固定长度的音频数据集
# ============================================================
def preprocess_audio(item, config):
    """预处理单个音频样本，确保固定长度"""
    audio = item['audio']
    y = audio['array']
    
    # 处理空音频
    if len(y) == 0:
        return np.zeros(config.audio_length, dtype=np.float32), item['label']
    
    # 处理多声道音频
    if y.ndim > 1:
        y = np.mean(y, axis=0)
    
    # 重采样
    if audio['sampling_rate'] != config.sample_rate:
        y = librosa.resample(
            y, 
            orig_sr=audio['sampling_rate'], 
            target_sr=config.sample_rate
        )
    
    # 确保音频长度一致
    if len(y) < config.audio_length:
        # 短音频填充
        y = np.pad(y, (0, config.audio_length - len(y)), mode='constant')
    elif len(y) > config.audio_length:
        # 随机裁剪（训练时）或中心裁剪（验证/测试时）
        start = random.randint(0, len(y) - config.audio_length)
        y = y[start:start+config.audio_length]
    
    # 最终强制长度匹配
    if len(y) != config.audio_length:
        # 使用截断或填充确保长度完全匹配
        if len(y) > config.audio_length:
            y = y[:config.audio_length]
        else:
            y = np.pad(y, (0, config.audio_length - len(y)), mode='constant')
    
    return y.astype(np.float32), item['label']

def create_processed_dataset(dataset, config, save_path):
    """创建并保存预处理后的数据集"""
    processed_data = []
    issues = 0
    
    for i in tqdm(range(len(dataset)), desc="处理音频样本"):
        try:
            item = dataset[i]
            waveform, label = preprocess_audio(item, config)
            
            # 检查长度
            if len(waveform) != config.audio_length:
                issues += 1
                # 强制修正长度
                if len(waveform) > config.audio_length:
                    waveform = waveform[:config.audio_length]
                else:
                    waveform = np.pad(waveform, (0, config.audio_length - len(waveform)), 
                                     mode='constant')
            
            processed_data.append({
                "waveform": waveform,
                "label": label
            })
        except Exception as e:
            print(f"处理样本 {i} 时出错: {str(e)}")
            # 添加静音样本
            processed_data.append({
                "waveform": np.zeros(config.audio_length, dtype=np.float32),
                "label": 0
            })
    
    print(f"处理完成! 共 {len(processed_data)} 个样本, {issues} 个样本需要长度修正")
    
    # 保存为Parquet文件
    df = pd.DataFrame(processed_data)
    df.to_parquet(save_path)
    print(f"预处理后的数据集已保存至: {save_path}")
    
    return df

class HTSAT_Swin_Transformer(paddle.nn.Layer):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # 频谱图生成层
        self.spectrogram = paddle.nn.Sequential(
            paddle.nn.Conv1D(
                in_channels=1, 
                out_channels=64, 
                kernel_size=config.n_fft, 
                stride=config.hop_length
            ),
            paddle.nn.ReLU(),
            paddle.nn.Dropout(config.htsat_drop_rate)
        )
        
        # 线性投影层（将64维特征投影到htsat_dim）
        self.projection = paddle.nn.Linear(64, config.htsat_dim)
        
        # Swin Transformer 块
        self.swin_blocks = paddle.nn.LayerList([
            self._make_swin_block(
                dim=config.htsat_dim, 
                num_heads=config.htsat_num_head,
                window_size=config.htsat_window_size,
                mlp_ratio=config.htsat_mlp_ratio,
                qkv_bias=config.htsat_qkv_bias,
                drop=config.htsat_drop_rate,
                attn_drop=config.htsat_attn_drop_rate
            )
            for _ in range(config.htsat_depth)
        ])
        
        # 分类头（维度将在第一次前向传播时确定）
        self.adaptive_pool = paddle.nn.AdaptiveAvgPool1D(1)
        self.flatten = paddle.nn.Flatten()
        self.classifier = None  # 稍后初始化
        
    def _make_swin_block(self, dim, num_heads, window_size, mlp_ratio=4., 
                         qkv_bias=True, drop=0., attn_drop=0.):
        """创建Swin Transformer块"""
        return paddle.nn.Sequential(
            paddle.nn.LayerNorm(dim),
            ShiftedWindowAttention(
                dim, 
                window_size=window_size, 
                num_heads=num_heads,
                qkv_bias=qkv_bias,
                attn_drop=attn_drop,
                proj_drop=drop
            ),
            paddle.nn.Linear(dim, dim),
            paddle.nn.GELU(),
            paddle.nn.Dropout(drop)
        )
    
    def forward(self, x):
        # 输入形状: (batch_size, 1, audio_length)
        
        # 生成频谱图
        x = self.spectrogram(x)  # 输出形状: (batch_size, 64, conv_length)
        
        # 调整维度顺序: (batch, channels, length) -> (batch, length, channels)
        x = x.transpose([0, 2, 1])
        
        # 投影到更高维度
        x = self.projection(x)  # 输出形状: (batch_size, conv_length, htsat_dim)
        
        # 通过Swin Transformer块
        for block in self.swin_blocks:
            x = block(x)
        
        # 调整维度顺序: (batch, length, channels) -> (batch, channels, length)
        x = x.transpose([0, 2, 1])
        
        # 全局平均池化
        x = self.adaptive_pool(x)  # 输出形状: (batch_size, htsat_dim, 1)
        
        # 展平
        x = self.flatten(x)  # 输出形状: (batch_size, htsat_dim)
        
        # 动态初始化分类头
        if self.classifier is None:
            self.classifier = paddle.nn.Linear(x.shape[1], self.config.num_classes)
            print(f"分类头输入维度自动设置为: {x.shape[1]}")
        
        # 分类
        logits = self.classifier(x)
        return logits

class ShiftedWindowAttention(paddle.nn.Layer):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        
        # 线性层
        self.qkv = paddle.nn.Linear(dim, dim * 3, bias_attr=qkv_bias)
        self.attn_drop = paddle.nn.Dropout(attn_drop)
        self.proj = paddle.nn.Linear(dim, dim)
        self.proj_drop = paddle.nn.Dropout(proj_drop)
    
    def forward(self, x):
        B, L, C = x.shape
        
        # 确保长度可以被窗口大小整除
        pad_len = (self.window_size - L % self.window_size) % self.window_size
        if pad_len > 0:
            x = paddle.nn.functional.pad(x, [0, 0, 0, pad_len], data_format="NLC")
            L = L + pad_len
        
        # 重塑为窗口 [B, num_windows, window_size, C]
        x = x.reshape([B, L // self.window_size, self.window_size, C])
        
        # 生成QKV
        qkv = self.qkv(x)
        qkv = qkv.reshape([B, L // self.window_size, self.window_size, 3, self.num_heads, C // self.num_heads])
        qkv = qkv.transpose([3, 0, 1, 4, 2, 5])  # [3, B, num_windows, num_heads, window_size, head_dim]
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # 计算注意力
        attn = (q @ k.transpose([0, 1, 2, 4, 3])) * self.scale
        attn = paddle.nn.functional.softmax(attn, axis=-1)
        attn = self.attn_drop(attn)
        
        # 输出 (简化维度转换)
        x = (attn @ v)  # 形状: [B, num_windows, num_heads, window_size, head_dim]
        
        # 合并多头输出
        x = x.transpose([0, 1, 3, 2, 4])  # [B, num_windows, window_size, num_heads, head_dim]
        x = x.reshape([B, L // self.window_size, self.window_size, C])  # 合并最后两个维度
        
        # 投影
        x = self.proj(x)
        x = self.proj_drop(x)
        
        # 恢复原始形状
        x = x.reshape([B, L, C])
        if pad_len > 0:
            x = x[:, :L-pad_len, :]
        
        return x
 
# 创建并保存预处理数据集
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))

train_dataset = full_dataset.select(range(train_size))
val_dataset = full_dataset.select(range(train_size, train_size + val_size))
test_dataset = full_dataset.select(range(train_size + val_size, len(full_dataset)))

print("创建训练集...")
train_df = create_processed_dataset(train_dataset, config, "processed_train.parquet")

print("创建验证集...")
val_df = create_processed_dataset(val_dataset, config, "processed_val.parquet")

print("创建测试集...")
test_df = create_processed_dataset(test_dataset, config, "processed_test.parquet")

# ============================================================
# 构建PaddlePaddle数据集 (使用预处理后的数据)
# ============================================================
class FixedAudioDataset(paddle.io.Dataset):
    def __init__(self, dataframe):
        self.data = dataframe
        print(f"数据集样本数: {len(dataframe)}")
        
    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        waveform = paddle.to_tensor(item['waveform'], dtype='float32')
        label = paddle.to_tensor([item['label']], dtype='int64')
        return waveform, label
    
    def __len__(self):
        return len(self.data)

# 加载预处理后的数据集
train_ds = FixedAudioDataset(train_df)
val_ds = FixedAudioDataset(val_df)
test_ds = FixedAudioDataset(test_df)

# ============================================================
# 创建DataLoader
# ============================================================
train_loader = paddle.io.DataLoader(
    train_ds, 
    batch_size=config.batch_size, 
    shuffle=True,
    drop_last=True,
    num_workers=config.num_workers
)

val_loader = paddle.io.DataLoader(
    val_ds, 
    batch_size=config.batch_size, 
    shuffle=False,
    num_workers=config.num_workers
)

test_loader = paddle.io.DataLoader(
    test_ds, 
    batch_size=config.batch_size, 
    shuffle=False,
    num_workers=config.num_workers
)

print(f"训练集批次数量: {len(train_loader)}")
print(f"验证集批次数量: {len(val_loader)}")
print(f"测试集批次数量: {len(test_loader)}")

# ============================================================
# 验证数据集长度一致性
# ============================================================
def verify_model_shapes(model, config):
    """验证模型各层输入输出形状"""
    print("\n=== 模型形状验证 ===")
    
    # 创建测试输入
    test_input = paddle.randn([2, 1, config.audio_length])
    print(f"输入形状: {test_input.shape}")
    
    # 频谱图层
    x = model.spectrogram(test_input)
    print(f"频谱图层输出: {x.shape}")
    
    # 转置和投影
    x = x.transpose([0, 2, 1])
    print(f"转置后: {x.shape}")
    
    x = model.projection(x)
    print(f"投影层输出: {x.shape}")
    
    # Swin块
    for i, block in enumerate(model.swin_blocks):
        x = block(x)
        print(f"Swin块 {i+1} 输出: {x.shape}")
    
    # 池化和分类
    x = x.transpose([0, 2, 1])
    print(f"池化前转置: {x.shape}")
    
    x = model.adaptive_pool(x)
    print(f"池化层输出: {x.shape}")
    
    x = model.flatten(x)
    print(f"展平层输出: {x.shape}")
    
    # 初始化分类头
    if model.classifier is None:
        model.classifier = paddle.nn.Linear(x.shape[1], config.num_classes)
        print(f"分类头输入维度自动设置为: {x.shape[1]}")
    
    x = model.classifier(x)
    print(f"分类层输出: {x.shape}")
    print("==================\n")



# ============================================================
# 模型训练代码 (示例)
# ============================================================
def train_model():
    # 创建模型
    model = HTSAT_Swin_Transformer(config)
    
    # 打印模型结构
    print("模型结构:")
    print(model)
    
    # 验证模型形状
    verify_model_shapes(model, config)
    
    # 损失函数
    criterion = paddle.nn.CrossEntropyLoss()
    
    # 创建优化器
    optimizer = paddle.optimizer.Adam(
        learning_rate=config.learning_rate,
        parameters=model.parameters(),
        weight_decay=config.weight_decay
    )
    
    # 学习率调度器
    scheduler = paddle.optimizer.lr.ReduceOnPlateau(
        learning_rate=config.learning_rate,
        mode='max',  # 监控验证准确率
        factor=0.5,  # 当准确率不再提升时，学习率乘以0.5
        patience=2,  # 连续2个epoch准确率没有提升时降低学习率
        threshold=0.001,  # 变化阈值
        cooldown=1,  # 冷却期
        min_lr=1e-6,  # 最小学习率
        verbose=True
    )
    
    # 创建模型保存目录
    model_dir = "best_models"
    os.makedirs(model_dir, exist_ok=True)
    
    # 初始化最佳准确率
    best_val_acc = 0.0
    best_epoch = 0
    
    print("开始训练...")
    for epoch in range(config.num_epochs):  # 使用配置中的epoch数
        # 训练阶段
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (waveforms, labels) in enumerate(train_loader):
            # 添加通道维度
            waveforms = waveforms.unsqueeze(1)
            
            # 前向传播
            logits = model(waveforms)
            loss = criterion(logits, labels.flatten())
            
            # 反向传播
            loss.backward()
            optimizer.step()
            optimizer.clear_grad()
            
            # 计算准确率
            preds = paddle.argmax(logits, axis=1)
            train_correct += (preds == labels.flatten()).sum().item()
            train_total += labels.shape[0]
            train_loss += loss.item()
            
            if batch_idx % 10 == 0:
                batch_acc = (preds == labels.flatten()).astype('float32').mean().item()
                print(f"Epoch {epoch+1}/{config.num_epochs}, Batch {batch_idx}/{len(train_loader)}, "
                      f"Loss: {loss.item():.4f}, Acc: {batch_acc:.4f}")
        
        train_acc = train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)
        
        # 验证阶段
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with paddle.no_grad():
            for waveforms, labels in val_loader:
                waveforms = waveforms.unsqueeze(1)
                logits = model(waveforms)
                loss = criterion(logits, labels.flatten())
                
                preds = paddle.argmax(logits, axis=1)
                val_correct += (preds == labels.flatten()).sum().item()
                val_total += labels.shape[0]
                val_loss += loss.item()
        
        val_acc = val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        
        # 打印epoch结果
        print(f"\nEpoch {epoch+1}/{config.num_epochs} 结果:")
        print(f"训练集 - 损失: {avg_train_loss:.4f}, 准确率: {train_acc:.4f}")
        print(f"验证集 - 损失: {avg_val_loss:.4f}, 准确率: {val_acc:.4f}")
        
        # 更新学习率 - 修复了调度器使用方式
        current_lr = optimizer.get_lr()
        scheduler.step(val_acc)  # 更新调度器状态
        new_lr = scheduler.get_lr()  # 获取更新后的学习率
        #optimizer.set_lr(new_lr)  # 设置优化器的新学习率
        print(f"学习率更新: {current_lr:.6f} -> {new_lr:.6f}")
        
        # 保存最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            
            # 保存模型参数
            model_path = os.path.join(model_dir, f"best_model_epoch_{epoch+1}_acc_{val_acc:.4f}.pdparams")
            paddle.save(model.state_dict(), model_path)
            
            # 保存优化器状态（可选）
            optim_path = os.path.join(model_dir, f"best_optimizer.pdopt")
            paddle.save(optimizer.state_dict(), optim_path)
            
            print(f"保存新的最佳模型! 验证准确率: {val_acc:.4f}")
        
        # 添加分隔线
        print("-" * 60)
    
    # 训练结束，加载最佳模型进行最终评估
    print(f"\n训练完成! 最佳模型来自第 {best_epoch} 个epoch, 验证准确率: {best_val_acc:.4f}")
    
    # 加载最佳模型
    model_files = [f for f in os.listdir(model_dir) if f.startswith('best_model') and f.endswith('.pdparams')]
    
    if model_files:
        # 按准确率排序（从高到低）
        model_files_sorted = sorted(
            model_files,
            key=lambda x: float(x.split("_")[-1].replace('.pdparams', '')),
            reverse=True
        )
        
        # 选择准确率最高的模型
        best_model_file = model_files_sorted[0]
        best_model_path = os.path.join(model_dir, best_model_file)
        model.set_state_dict(paddle.load(best_model_path))
        print(f"加载最佳模型: {best_model_path}")
    else:
        print("警告: 未找到最佳模型文件!")
    
    # 在测试集上评估
    model.eval()
    test_correct = 0
    test_total = 0
    
    with paddle.no_grad():
        for waveforms, labels in test_loader:
            waveforms = waveforms.unsqueeze(1)
            logits = model(waveforms)
            preds = paddle.argmax(logits, axis=1)
            test_correct += (preds == labels.flatten()).sum().item()
            test_total += labels.shape[0]
    
    test_acc = test_correct / test_total
    print(f"测试集准确率: {test_acc:.4f}")
    
    # 保存最终模型
    final_model_path = os.path.join(model_dir, "final_model.pdparams")
    paddle.save(model.state_dict(), final_model_path)
    print(f"最终模型已保存至: {final_model_path}")
    
    # 保存完整模型（包含结构）
    full_model_path = os.path.join(model_dir, "full_model")
    paddle.jit.save(model, full_model_path)
    print(f"完整模型（含结构）已保存至: {full_model_path}")
    
    return model, best_val_acc, test_acc


# 运行训练
train_model()

-------------加载原始数据集-------------
创建训练集...


处理音频样本: 100%|██████████| 1736/1736 [00:01<00:00, 1664.36it/s]


处理完成! 共 1736 个样本, 0 个样本需要长度修正
预处理后的数据集已保存至: processed_train.parquet
创建验证集...


处理音频样本: 100%|██████████| 217/217 [00:00<00:00, 1710.43it/s]

处理完成! 共 217 个样本, 0 个样本需要长度修正


...

处理音频样本: 100%|██████████| 218/218 [00:00<00:00, 1638.12it/s]

处理完成! 共 218 个样本, 0 个样本需要长度修正



W0605 18:06:22.982496   231 gpu_resources.cc:306] WARNING: device: 0. The installed Paddle is compiled with CUDNN 8.9, but CUDNN version in your machine is 8.9, which may cause serious incompatible bug. Please recompile or reinstall Paddle with compatible CUDNN version.


预处理后的数据集已保存至: processed_test.parquet
数据集样本数: 1736
数据集样本数: 217
数据集样本数: 218
训练集批次数量: 54
验证集批次数量: 7
测试集批次数量: 7
模型结构:
HTSAT_Swin_Transformer(
  (spectrogram): Sequential(
    (0): Conv1D(1, 64, kernel_size=[1024], stride=[256], data_format=NCL)
    (1): ReLU()
    (2): Dropout(p=0.1, axis=None, mode=upscale_in_train)
  )
  (projection): Linear(in_features=64, out_features=128, dtype=float32)
  (swin_blocks): LayerList(
    (0): Sequential(
      (0): LayerNorm(normalized_shape=[128], epsilon=1e-05)
      (1): ShiftedWindowAttention(
        (qkv): Linear(in_features=128, out_features=384, dtype=float32)
        (attn_drop): Dropout(p=0.1, axis=None, mode=upscale_in_train)
        (proj): Linear(in_features=128, out_features=128, dtype=float32)
        (proj_drop): Dropout(p=0.1, axis=None, mode=upscale_in_train)
      )
      (2): Linear(in_features=128, out_features=128, dtype=float32)
      (3): GELU(approximate=False)
      (4): Dropout(p=0.1, axis=None, mode=upscale_in_train)
    )
   

NotImplementedError: 

In [11]:
# import basic packages
import os
import numpy as np
import sys
import paddle
import paddle.distributed as dist
from paddle.io import DataLoader, DistributedBatchSampler
from paddle.optimizer import AdamW
import warnings
import subprocess
import glob
import pandas as pd
from tqdm import tqdm
import random
import json
import librosa
from datasets import load_dataset

# 设置GPU
paddle.set_device('gpu:0')

# 构建工作空间并下载必要文件
def create_path(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

# 基础路径配置
base_path = "/home/aistudio/data"
workspace = os.path.join(base_path, "audio_workspace")
dataset_path = os.path.join(workspace, "custom_dataset")
checkpoint_path = os.path.join(workspace, "ckpt")

create_path(workspace)
create_path(dataset_path)
create_path(checkpoint_path)

# 下载预训练权重 (这里需要替换为实际的PaddlePaddle预训练权重)
pretrain_ckpt_path = "" #os.path.join(checkpoint_path, 'htsat_audioset_pretrain.pdparams')

# ============================================================
# 加载数据集
# ============================================================
print("-------------加载Hemg/audio-based-violence-dataset数据集-------------")
# 加载单个文件
ds = load_dataset('parquet', data_files='/home/aistudio/train-00000-of-00001.parquet')

# 创建标签映射
label_map = {0: "non_violence", 1: "violence"}
dataset_path = "/home/aistudio/data"  # 确保定义这个路径

# 保存标签映射
with open(os.path.join(dataset_path, "label_map.json"), 'w') as f:
    json.dump(label_map, f)

# 由于只有一个拆分，我们手动划分数据集
full_dataset = ds['train']  # 提取唯一的数据集拆分
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))

# 划分数据集
train_dataset = full_dataset.select(range(train_size))
val_dataset = full_dataset.select(range(train_size, train_size + val_size))
test_dataset = full_dataset.select(range(train_size + val_size, len(full_dataset)))

print(f"训练集样本数: {len(train_dataset)}")
print(f"验证集样本数: {len(val_dataset)}")
print(f"测试集样本数: {len(test_dataset)}")

# ============================================================
# 构建PaddlePaddle数据集
# ============================================================
class AudioDataset(paddle.io.Dataset):
    def __init__(self, dataset, config, mode='train'):
        self.dataset = dataset
        self.config = config
        self.mode = mode
        self.audio_length = config.audio_length
        
        print(f"{mode}数据集: {len(dataset)} 个样本")
        self.length_issues = []  # 记录长度问题的样本
        
    def __getitem__(self, idx):
        try:
            item = self.dataset[idx]
            
            # 获取音频数据
            audio = item['audio']
            y = audio['array']
            
            # 处理空音频
            if len(y) == 0:
                y = np.zeros(self.audio_length)
                if idx not in self.length_issues:
                    print(f"警告: 空音频样本 (索引 {idx}), 使用静音替代")
                    self.length_issues.append(idx)
            
            # 处理多声道音频
            if y.ndim > 1:
                y = np.mean(y, axis=0)
            
            # 重采样
            if audio['sampling_rate'] != self.config.sample_rate:
                y = librosa.resample(
                    y, 
                    orig_sr=audio['sampling_rate'], 
                    target_sr=self.config.sample_rate
                )
            
            # 确保音频长度一致（核心修复）
            if len(y) < self.audio_length:
                # 短音频填充
                y = np.pad(y, (0, self.audio_length - len(y)), mode='constant')
            elif len(y) > self.audio_length:
                # 长音频裁剪
                if self.mode == 'train':
                    start = random.randint(0, len(y) - self.audio_length)
                else:
                    start = (len(y) - self.audio_length) // 2  # 验证/测试集使用中心裁剪
                y = y[start:start+self.audio_length]
            
            # 最终强制长度匹配（双重保险）
            if len(y) != self.audio_length:
                if idx not in self.length_issues:
                    print(f"警告: 样本 {idx} 长度 {len(y)} ≠ 目标长度 {self.audio_length}, 强制调整")
                    self.length_issues.append(idx)
                
                # 使用截断或填充确保长度完全匹配
                if len(y) > self.audio_length:
                    y = y[:self.audio_length]
                else:
                    y = np.pad(y, (0, self.audio_length - len(y)), mode='constant')
            
            # 转换为Paddle Tensor
            waveform = paddle.to_tensor(y, dtype='float32')
            label = paddle.to_tensor([item['label']], dtype='int64')
            
            return waveform, label
        
        except Exception as e:
            print(f"处理样本 {idx} 时出错: {str(e)}")
            # 返回静音样本作为后备
            return paddle.zeros([self.audio_length]), paddle.to_tensor([0])
    
    def __len__(self):
        return len(self.dataset)
         

# ============================================================
# 模型配置
# ============================================================
class Config:
    def __init__(self):
        # 音频参数
        self.sample_rate = 16000
        self.audio_length = 5 * self.sample_rate  # 5秒音频
        
        # 模型参数
        self.num_classes = 2  # 二分类：暴力和非暴力
        self.htsat_spec_size = 256
        self.htsat_patch_size = 4
        self.htsat_window_size = 8
        self.htsat_depth = 2
        self.htsat_dim = 128
        self.htsat_stride = 4
        self.htsat_num_head = 4
        
        # 训练参数
        self.batch_size = 32
        self.learning_rate = 1e-4
        self.weight_decay = 1e-5
        self.max_epoch = 50
        self.num_workers = 4
        self.log_interval = 20

config = Config()


# ============================================================
# HTSAT模型 (PaddlePaddle实现)
# ============================================================
class HTSAT_Swin_Transformer(paddle.nn.Layer):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # 频谱图生成
        self.spectrogram = paddle.nn.Sequential(
            paddle.nn.Conv1D(1, 64, kernel_size=1024, stride=256),
            paddle.nn.ReLU(),
            paddle.nn.Dropout(0.2)
        )
        
        # Swin Transformer 块
        self.swin_blocks = paddle.nn.LayerList([
            self._make_swin_block(dim=config.htsat_dim, 
                                  num_heads=config.htsat_num_head,
                                  window_size=config.htsat_window_size)
            for _ in range(config.htsat_depth)
        ])
        
        # 分类头
        self.classifier = paddle.nn.Sequential(
            paddle.nn.AdaptiveAvgPool1D(1),
            paddle.nn.Flatten(),
            paddle.nn.Linear(config.htsat_dim, config.num_classes)
        )
    
    def _make_swin_block(self, dim, num_heads, window_size):
        return paddle.nn.Sequential(
            paddle.nn.LayerNorm(dim),
            paddle.nn.MultiHeadAttention(dim, num_heads),
            paddle.nn.LayerNorm(dim),
            paddle.nn.Linear(dim, dim * 4),
            paddle.nn.GELU(),
            paddle.nn.Linear(dim * 4, dim),
            paddle.nn.Dropout(0.1)
        )
    
    def forward(self, x):
        # x: [batch, 1, audio_length]
        
        # 生成频谱图
        spec = self.spectrogram(x)  # [batch, 64, spec_length]
        spec = spec.transpose([0, 2, 1])  # [batch, spec_length, 64]
        
        # 通过Swin Transformer块
        for block in self.swin_blocks:
            spec = block(spec)
        
        # 分类
        logits = self.classifier(spec)
        return logits

# ============================================================
# 训练函数
# ============================================================
def train_model():
    # 初始化分布式训练
    dist.init_parallel_env()
    # 创建Paddle数据集实例
    train_ds = AudioDataset(train_dataset, config, mode='train')
    val_ds = AudioDataset(val_dataset, config, mode='validation')
    test_ds = AudioDataset(test_dataset, config, mode='test')
    
    # 创建分布式采样器
    train_sampler = DistributedBatchSampler(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        drop_last=True
    )
    
    val_sampler = DistributedBatchSampler(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        drop_last=False
    )
    
    test_sampler = DistributedBatchSampler(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        drop_last=False
    )
    
    # 创建数据加载器
    train_loader = DataLoader(
        train_dataset,
        batch_sampler=train_sampler,
        num_workers=config.num_workers
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_sampler=val_sampler,
        num_workers=config.num_workers
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_sampler=test_sampler,
        num_workers=config.num_workers
    )
    
    # 创建模型
    model = HTSAT_Swin_Transformer(config)
    
    # 分布式训练
    if dist.get_world_size() > 1:
        model = paddle.DataParallel(model)
    
    # 定义优化器和损失函数
    clip = paddle.nn.ClipGradByGlobalNorm(clip_norm=1.0)
    optimizer = AdamW(
        parameters=model.parameters(),
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        grad_clip=clip
    )
    
    criterion = paddle.nn.CrossEntropyLoss()
    
    # 训练循环
    best_acc = 0.0
    for epoch in range(config.max_epoch):
        # 训练阶段
        model.train()
        total_loss = 0.0
        total_samples = 0
        correct_train = 0
        
        for batch_idx, (waveform, labels) in enumerate(train_loader):
            # 前向传播
            logits = model(waveform.unsqueeze(1))
            loss = criterion(logits, labels.flatten())
            
            # 反向传播
            loss.backward()
            optimizer.step()
            optimizer.clear_grad()
            
            # 计算准确率
            preds = paddle.argmax(logits, axis=1)
            correct_train += (preds == labels.flatten()).sum().numpy()[0]
            total_samples += labels.shape[0]
            
            total_loss += loss.numpy()[0]
            
            if batch_idx % config.log_interval == 0:
                batch_acc = (preds == labels.flatten()).astype('float32').mean().numpy()[0]
                print(f"Epoch {epoch+1}/{config.max_epoch}, "
                      f"Batch {batch_idx}/{len(train_loader)}, "
                      f"Loss: {loss.numpy()[0]:.4f}, "
                      f"Acc: {batch_acc:.4f}")
        
        avg_train_loss = total_loss / len(train_loader)
        train_acc = correct_train / total_samples
        
        # 验证阶段
        model.eval()
        correct_val = 0
        total_val = 0
        val_loss = 0.0
        
        with paddle.no_grad():
            for waveform, labels in val_loader:
                logits = model(waveform.unsqueeze(1))
                loss = criterion(logits, labels.flatten())
                
                preds = paddle.argmax(logits, axis=1)
                correct_val += (preds == labels.flatten()).sum().numpy()[0]
                total_val += labels.shape[0]
                val_loss += loss.numpy()[0]
        
        val_acc = correct_val / total_val
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1}/{config.max_epoch}, "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f}, "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")
        
        # 保存最佳模型
        if val_acc > best_acc:
            best_acc = val_acc
            save_path = os.path.join(checkpoint_path, f'best_model_{val_acc:.4f}.pdparams')
            paddle.save(model.state_dict(), save_path)
            print(f"保存最佳模型，验证准确率: {val_acc:.4f} 到 {save_path}")
    
    # 最终测试
    model.eval()
    correct_test = 0
    total_test = 0
    
    with paddle.no_grad():
        for waveform, labels in test_loader:
            logits = model(waveform.unsqueeze(1))
            preds = paddle.argmax(logits, axis=1)
            correct_test += (preds == labels.flatten()).sum().numpy()[0]
            total_test += labels.shape[0]
    
    test_acc = correct_test / total_test
    print(f"训练完成，测试准确率: {test_acc:.4f}")
    
    # 保存最终模型
    final_save_path = os.path.join(checkpoint_path, f'final_model_{test_acc:.4f}.pdparams')
    paddle.save(model.state_dict(), final_save_path)
    print(f"保存最终模型到 {final_save_path}")

# ============================================================
# 主函数
# ============================================================
if __name__ == '__main__':
    train_model()

# ============================================================
# 推理函数
# ============================================================
class AudioClassifier:
    def __init__(self, model_path, config):
        self.config = config
        self.device = paddle.get_device()
        
        # 加载模型
        self.model = HTSAT_Swin_Transformer(config)
        state_dict = paddle.load(model_path)
        self.model.set_state_dict(state_dict)
        self.model.eval()
        
        # 加载标签映射
        label_map_path = os.path.join(dataset_path, "label_map.json")
        if os.path.exists(label_map_path):
            with open(label_map_path, 'r') as f:
                self.label_map = json.load(f)
        else:
            self.label_map = {0: "non_violence", 1: "violence"}
    
    def predict(self, audio_path):
        try:
            # 加载音频文件
            y, sr = librosa.load(audio_path, sr=self.config.sample_rate, mono=True)
            
            # 确保音频长度一致
            if len(y) < self.config.audio_length:
                # 填充
                y = np.pad(y, (0, self.config.audio_length - len(y)), mode='constant')
            elif len(y) > self.config.audio_length:
                # 随机裁剪
                start = random.randint(0, len(y) - self.config.audio_length)
                y = y[start:start+self.config.audio_length]
            
            # 转换为Paddle Tensor
            waveform = paddle.to_tensor(y, dtype='float32').unsqueeze(0).unsqueeze(0)
            
            # 推理
            with paddle.no_grad():
                logits = self.model(waveform)
                probs = paddle.nn.functional.softmax(logits, axis=1).numpy()[0]
                pred_label = np.argmax(probs)
                pred_prob = probs[pred_label]
            
            return pred_label, pred_prob, self.label_map[str(pred_label)]
        except Exception as e:
            print(f"处理音频 {audio_path} 时出错: {e}")
            return None, None, None

# # 示例推理
# if len(ds['test']) > 0:
#     print("\n-------------测试推理-------------")
#     # 使用训练保存的最佳模型
#     best_model_path = glob.glob(os.path.join(checkpoint_path, 'best_model_*.pdparams'))
#     if best_model_path:
#         best_model_path = sorted(best_model_path, reverse=True)[0]  # 取最新的最佳模型
#     else:
#         best_model_path = glob.glob(os.path.join(checkpoint_path, 'final_model_*.pdparams'))[0]
    
#     classifier = AudioClassifier(best_model_path, config)
    
#     # 随机选择一个测试样本
#     test_item = ds['test'][random.randint(0, len(ds['test'])-1]
#     audio_path = test_item['audio']['path']
#     true_label = test_item['label']
    
#     pred_label, pred_prob, label_name = classifier.predict(audio_path)
    
#     if pred_label is not None:
#         print(f"音频文件: {os.path.basename(audio_path)}")
#         print(f"真实标签: {true_label} ({label_map[true_label]})")
#         print(f"预测标签: {pred_label} ({label_name}), 置信度: {pred_prob:.4f}")

-------------加载Hemg/audio-based-violence-dataset数据集-------------
训练集样本数: 1736
验证集样本数: 217
测试集样本数: 218
train数据集: 1736 个样本
validation数据集: 217 个样本
test数据集: 218 个样本


======================= Modified FLAGS detected =======================
FLAGS(name='FLAGS_nccl_dir', current_value='/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/../nvidia/nccl/lib', default_value='')
FLAGS(name='FLAGS_cudnn_dir', current_value='/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/../nvidia/cudnn/lib', default_value='')
FLAGS(name='FLAGS_enable_pir_in_executor', current_value=True, default_value=False)
FLAGS(name='FLAGS_flagcx_dir', current_value='/build/lib', default_value='')
FLAGS(name='FLAGS_cublas_dir', current_value='/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/../nvidia/cublas/lib', default_value='')
FLAGS(name='FLAGS_cupti_dir', current_value='/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/../nvidia/cuda_cupti/lib', default_value='')
FLAGS(name='FLAGS_curand_dir', current_value='/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/.

SystemError: (Fatal) Blocking queue is killed because the data reader raises an exception.
  [Hint: Expected killed_ != true, but received killed_:1 == true:1.] (at ../paddle/phi/core/operators/reader/blocking_queue.h:175)


请点击[此处](https://ai.baidu.com/docs#/AIStudio_Project_Notebook/a38e5576)查看本环境基本用法.  <br>
Please click [here ](https://ai.baidu.com/docs#/AIStudio_Project_Notebook/a38e5576) for more detailed instructions. 